# Fabric 00 · Land the surgical cases into OneLake

**Foundry trains the model; Fabric operationalizes it.** This first notebook lands the synthetic Return-to-OR cases as a governed **Delta table** (`surgical_episodes`) in a Fabric **Lakehouse** — the single source the scoring, routing, and evaluation notebooks read from.

> **Dual-mode:** runs in Microsoft Fabric (Spark + Lakehouse) *or* locally (pandas + the repo's `fine-tuning/data/`). All data is fully **synthetic & PHI-free**.

**To run in Fabric:** attach a Lakehouse, then upload `fine-tuning/data/rtor_eval.jsonl` to the Lakehouse under `Files/rtor/`.

In [ ]:
# === Dual-mode setup: Microsoft Fabric (Spark + Lakehouse) OR local (pandas + repo files) ===
import os, json
from pathlib import Path

try:
    import notebookutils            # exists ONLY inside Microsoft Fabric
    IN_FABRIC = True
except Exception:
    IN_FABRIC = False

def _find_data_dir():
    here = Path.cwd()
    for c in [here, *here.parents]:
        d = c / 'fine-tuning' / 'data'
        if d.exists():
            return d
    return Path('fine-tuning/data')
DATA_DIR = None if IN_FABRIC else _find_data_dir()

AZURE_OPENAI_ENDPOINT = os.environ.get('AZURE_OPENAI_ENDPOINT', 'https://<your-foundry>.cognitiveservices.azure.com/')
API_VERSION           = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
BASE_DEPLOYMENT       = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')
TUNED_DEPLOYMENT      = os.environ.get('TUNED_DEPLOYMENT', 'acme-rtor-deployment')

# Entra token for Azure OpenAI: Fabric token broker in-cloud, DefaultAzureCredential locally.
if IN_FABRIC:
    def _token():
        return notebookutils.credentials.getToken('https://cognitiveservices.azure.com')
else:
    from azure.identity import DefaultAzureCredential
    _cred = DefaultAzureCredential()
    def _token():
        return _cred.get_token('https://cognitiveservices.azure.com/.default').token

from openai import AzureOpenAI
client = AzureOpenAI(
    azure_endpoint          = AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider = _token,
    api_version             = API_VERSION,
)
print('mode    :', 'FABRIC' if IN_FABRIC else 'LOCAL')
print('endpoint:', AZURE_OPENAI_ENDPOINT)
print('models  : base=' + BASE_DEPLOYMENT + '  tuned=' + TUNED_DEPLOYMENT)


---
## Step 1 — Read the synthetic source cases

In [ ]:
if IN_FABRIC:
    src = '/lakehouse/default/Files/rtor/rtor_eval.jsonl'
    cases = [json.loads(l) for l in open(src, encoding='utf-8') if l.strip()]
else:
    p = DATA_DIR / 'rtor_eval.jsonl'
    cases = [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]
print('source cases:', len(cases))
print('keys:', sorted(cases[0].keys()))


---
## Step 2 — Flatten to a table-friendly shape

Nested fields (timeline, progress note, op notes) are kept as a single `case_json` string so the table stays simple; flat columns drive BI filters and joins.

In [ ]:
rows = []
for c in cases:
    rows.append({
        'case_id'                        : c['case_id'],
        'provider_npi'                   : c.get('provider_npi'),
        'index_surgery_procedure_desc'   : c.get('index_surgery_procedure_desc'),
        'current_surgery_procedure_desc' : c.get('current_surgery_procedure_desc'),
        'gold_is_return_to_or'           : bool(c.get('gold_is_return_to_or')),
        'gold_evidence'                  : c.get('gold_evidence'),
        'case_json'                      : json.dumps(c),
    })
print('prepared', len(rows), 'rows')


---
## Step 3 — Write to OneLake (Delta) or a local stand-in

In Fabric this is a managed **Delta table** in the Lakehouse — queryable from Spark, the SQL endpoint, and Power BI. Locally we write a parquet/CSV stand-in just to prove the shape.

In [ ]:
if IN_FABRIC:
    sdf = spark.createDataFrame(rows)
    sdf.write.format('delta').mode('overwrite').saveAsTable('surgical_episodes')
    print('Lakehouse table surgical_episodes:', sdf.count(), 'rows')
    display(spark.read.table('surgical_episodes').limit(5))
else:
    import pandas as pd
    pdf = pd.DataFrame(rows)
    try:
        out = DATA_DIR / 'surgical_episodes.parquet'
        pdf.to_parquet(out); print('LOCAL stand-in ->', out)
    except Exception as e:
        out = DATA_DIR / 'surgical_episodes.csv'
        pdf.to_csv(out, index=False); print('LOCAL stand-in (csv) ->', out, '|', e)
    display(pdf.head())


---
## Takeaways

- One governed `surgical_episodes` table is the contract every downstream notebook reads.
- The same flat-plus-`case_json` shape works for Spark scoring **and** Power BI.
- Next: **Fabric 01** stands up the *cheap* RAG baseline — no training — to set the bar.